# ViT5 inference smoke test

Runs a small inference and ROUGE check against the raw dataset. The first run downloads the model from Hugging Face.

## Setup

The Colab runtime clones the repository into /content/thuc-tap when needed.

In [2]:
from pathlib import Path
import subprocess
import sys

PROJECT_SUBDIR = Path('tuan 3-4')
IN_COLAB = 'google.colab' in sys.modules

def is_project(path: Path) -> bool:
    return (path / 'src' / 'data.py').is_file()

# Clone the repository on a fresh Colab runtime.
COLAB_WORKSPACE = Path('/content/thuc-tap')
if IN_COLAB and not is_project(COLAB_WORKSPACE / PROJECT_SUBDIR):
    if COLAB_WORKSPACE.exists():
        raise RuntimeError(
            f'{COLAB_WORKSPACE} đã tồn tại nhưng không chứa project hợp lệ. ' 
            'Hãy xóa thư mục này trên runtime rồi chạy lại cell.'
        )
    subprocess.run(
        ['git', 'clone', 'https://github.com/dungcony/text-sumarization.git', str(COLAB_WORKSPACE)],
        check=True,
    )

# Support both Colab and local notebook working directories.
cwd = Path.cwd().resolve()
candidates = (
    cwd,
    cwd.parent,
    cwd / PROJECT_SUBDIR,
    COLAB_WORKSPACE / PROJECT_SUBDIR,
)
PROJECT_ROOT = next((path for path in candidates if is_project(path)), None)
if PROJECT_ROOT is None:
    tried = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Không tìm thấy src/. Đã kiểm tra:\n{tried}')

WORKSPACE_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f'[setup] project_root={PROJECT_ROOT}')
print(f'[setup] runtime={"colab" if IN_COLAB else "local"}')

PROJECT_ROOT : /content/thuc-tap/tuan 3-4/summarization
WORKSPACE_ROOT: /content/thuc-tap


In [3]:
# Install the project and runtime dependencies in Colab.
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
# else:
#     subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)

import torch
import transformers
import pandas
import rouge_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[setup] torch={torch.__version__} transformers={transformers.__version__}')
print(f'[setup] device={DEVICE}')

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Thiết bị sẽ dùng: cuda


## Load data

Use the prepared raw CSV and validate the records before inference.

In [4]:
from data import read_rows, records_from_rows

RAW_SOURCE = PROJECT_ROOT / 'data' / 'raw' / 'vietnews_medical_raw_1000.csv'

rows = read_rows(RAW_SOURCE)
records, audit = records_from_rows(
    rows,
    dataset_name='vietnews-medical',
    min_article_words=10,
    min_summary_words=3,
    keep_at_most=1000,
)
print(f"[data] input_rows={len(rows)} accepted_rows={len(records)}")
print(f"[data] rejected={audit['rejected_by_reason']}")
audit

Nguồn: 1000 bản ghi
Đầu ra: 990 cặp hợp lệ
Loại: {'exact_duplicate_pair': 10}


{'input_rows': 1000,
 'accepted_rows': 990,
 'rejected_rows': 10,
 'rejected_by_reason': {'exact_duplicate_pair': 10},
 'columns': {'article': 'Document', 'summary': 'Summary', 'id': None},
 'filters': {'min_article_words': 10,
  'min_summary_words': 3,
  'requires_summary_shorter_than_article': True,
  'deduplicate': 'exact article-summary pair'},
 'length_statistics_words': {'article_mean': 459.08,
  'article_min': 108,
  'article_max': 1525,
  'summary_mean': 37.49,
  'summary_min': 17,
  'summary_max': 80}}

In [5]:
import pandas as pd

raw_frame = pd.read_csv(RAW_SOURCE)
raw_frame[['id', 'article', 'summary', 'source_dataset']].head(3)

,id,article,summary,source_dataset
0,1,Đây là một trong những nội dung tại văn bản vừ...,"Các quận , huyện , thị xã tuyên truyền bằng nh...",vietnews-medical
1,2,"Chiều 12/3 , ông Vũ Hùng Triều , Trưởng phòng ...",Cháu bé được phát hiện trong tình trạng chưa c...,vietnews-medical
2,3,"Thoạt đầu nhìn vào bức ảnh , nếu không có dấu ...",Báo tuyết ( Panthera uncia ) được mệnh danh là...,vietnews-medical


## Run inference

Keep the sample size small for a fast integration check.

In [6]:
from model import generate_summaries
from metrics import compute_rouge, compression_statistics
from data import records_from_rows, read_rows

MODEL_NAME = 'VietAI/vit5-base-vietnews-summarization'
SAMPLES_PATH = RAW_SOURCE
MAX_SAMPLES = 5

sample_records, sample_audit = records_from_rows(
    read_rows(SAMPLES_PATH), keep_at_most=MAX_SAMPLES
)
articles = [record.article for record in sample_records]
references = [record.summary for record in sample_records]

predictions, used_device = generate_summaries(
    articles,
    model_name=MODEL_NAME,
    device='auto',
    prefix='summarize: ',
    batch_size=1,
    max_source_length=384,
    max_new_tokens=64,
    min_new_tokens=8,
    num_beams=2,
    no_repeat_ngram_size=3,
)

print(f'[inference] samples={len(predictions)} device={used_device} model={MODEL_NAME}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

Đã chạy 5 mẫu trên cuda.


In [7]:
rouge = compute_rouge(predictions, references)
compression = compression_statistics(articles, predictions)

score_table = pd.DataFrame(
    {metric: values for metric, values in rouge.items()}
).T.rename(columns={'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1'})
display(score_table)
print(f'[metrics] rouge1_f1={rouge["rouge1"]["f1"]:.2f} rouge2_f1={rouge["rouge2"]["f1"]:.2f} rougeL_f1={rouge["rougeL"]["f1"]:.2f}')
print(f'[metrics] compression={compression["compression_percent"]:.2f}%')

,Precision,Recall,F1
rouge1,67.29,51.80,58.05
rouge2,25.12,19.21,21.57
rougeL,39.74,30.62,34.29


Thống kê độ dài: {'source_words_mean': 96.8, 'summary_words_mean': 28.6, 'compression_percent': 70.45}


In [8]:
prediction_frame = pd.DataFrame(
    {
        'id': [record.id for record in sample_records],
        'article': articles,
        'reference': references,
        'prediction': predictions,
    }
)
pd.set_option('display.max_colwidth', 180)
prediction_frame[['id', 'reference', 'prediction']]

,id,reference,prediction
0,1,"Ngủ đủ 7-8 tiếng mỗi đêm giúp não bộ dọn dẹp chất thải chuyển hóa, củng cố trí nhớ và giảm nguy cơ suy giảm trí nhớ, mất tập trung.",Người thường thiếu ngủ có nguy cơ mắc các bệnh suy giảm trí nhớ và khó tập trung hơn những người ngủ đủ giấc .
1,2,"AI đang cách mạng hóa ngành y tế nhờ khả năng phân tích dữ liệu nhanh chóng và chẩn đoán chính xác ung thư, hứa hẹn trở thành công cụ hỗ trợ đắc lực cho bác sĩ.",Các nhà khoa học cho biết trí tuệ nhân tạo có khả năng phát hiện sớm và thậm chí vượt trội hơn so với các bác sĩ chuyên khoa giàu kinh nghiệm .
2,3,"Biến đổi khí hậu làm tăng nhiệt độ, tan băng, nước biển dâng và gây ra nhiều thời tiết cực đoan, đe dọa nghiêm trọng đến đời sống con người và hệ sinh thái.",PGS.TS Nguyễn Ngọc Long - Đại học Quốc gia Hà Nội đã đưa ra nhận định tác động của biến đổi khí hậu đang ảnh hưởng nghiêm trọng đến hệ sinh thái toàn cầu .
3,4,"Chế độ ăn Địa Trung Hải ưu tiên rau củ, dầu oliu, hải sản và hạn chế thịt đỏ giúp tăng tuổi thọ, giảm nguy cơ mắc bệnh tim mạch và tiểu đường.",Chế độ ăn Địa Trung Hải là một trong những lối sống lành mạnh nhất thế giới .
4,5,"Kính viễn vọng James Webb gửi về hình ảnh chi tiết của cụm thiên hà xa xôi, giúp tìm hiểu vũ trụ sơ khai và phát hiện dấu hiệu có thể có sự sống trên các ngoại hành tinh.",Cơ quan Hàng không vũ trụ Mỹ ( NASA ) vừa công bố những hình ảnh sắc nét về một cụm thiên hà cách Trái Đất hàng tỷ năm ánh sáng .


## Save run artifact

Writes metrics, generation parameters, and predictions as JSON.

In [9]:
import json
from datetime import datetime, timezone

NOTEBOOK_RESULT = PROJECT_ROOT / 'results' / 'notebook_vit5_sample_5.json'
payload = {
    'run': {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model': MODEL_NAME,
        'device': used_device,
        'number_of_examples': len(sample_records),
    },
    'generation': {
        'prefix': 'summarize: ', 'max_source_length': 384,
        'max_new_tokens': 64, 'min_new_tokens': 8,
        'num_beams': 2, 'no_repeat_ngram_size': 3,
    },
    'metrics': {'rouge': rouge, 'compression': compression},
    'predictions': prediction_frame.to_dict(orient='records'),
}
NOTEBOOK_RESULT.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'[output] wrote={NOTEBOOK_RESULT}')

Đã lưu: /content/thuc-tap/tuan 3-4/summarization/results/notebook_vit5_sample_5.json


## Notes

This is an inference smoke test, not a final evaluation. Use a fixed held-out split for model comparison.